# Model A — Retraining Pipeline
### Jianhui — Tree-Based Track (XGBoost + SMOTE)

**Why this is one notebook, not two.** EDA and Feature Engineering represent a
decision — *which* features matter and *how* they're safely transformed —
made once by a person, validated against evidence, and not something that
should silently re-run just because a new data file shows up. What genuinely
needs to re-run whenever new data arrives is everything downstream of that
decision: re-apply the same transforms, retrain, evaluate, and decide whether
the result is good enough to replace what's currently deployed.

This notebook is that downstream piece, parameterized by which raw data file
to use, so **the same notebook is re-run for every retraining cycle** rather
than a second notebook duplicating logic that already exists in Feature
Engineering and the Model A Final notebook. Duplicating that logic in a
second place is exactly the class of bug this project already hit once
(Feature Engineering and Model A silently diverging) — this notebook avoids
that by treating `eda_feature_candidates.json` and
`feature_engineering_manifest.json` as the source of truth for *which*
features and transforms to apply, not by re-deciding anything itself.

**⚠ Important note on the transformation code below:** the feature-engineering
formulas in Section 3 (Haversine distance, log transforms, age, temporal
features) and the target-encoding logic in Section 5 are written to match the
patterns already established in your Feature Engineering and Model A Final
notebooks. Before relying on this notebook, **diff Section 3 and Section 5
cell-by-cell against your actual Feature Engineering notebook** to confirm
the formulas are character-for-character identical — this notebook
reconstructs that logic rather than importing it directly, since everything
in this project currently lives in notebooks rather than shared, importable
`.py` modules. Closing that gap properly (a shared `feature_transforms.py`
both notebooks import) is the real long-term fix — see the closing note in
Section 9.

**What this notebook does, in order:**
1. Load new raw data (parameterized — not hard-coded to one file)
2. Re-apply Feature Engineering's transforms, reading its manifest as the
   source of truth for which features to compute
3. Split, out-of-fold target-encode, SMOTE (identical methodology to the
   locked pipeline — nothing re-decided here)
4. Retrain using the **already-locked final hyperparameters and feature set**
   — retraining re-fits on new data, it does not re-run hyperparameter
   search
5. Evaluate the new candidate on a held-out split of the new data
6. **Fetch the currently deployed model's metrics from the MLflow Model
   Registry** (whatever the `champion-candidate` alias currently points at)
7. **Compare candidate vs. current champion and decide** — only register and
   promote the alias if the candidate is actually better; otherwise, log the
   comparison and leave the current champion in place
8. Export the same artifact set as before (preprocessing bundle, deployment
   contract, results manifest) so nothing downstream needs to change


In [1]:
# ============================================================
# Install required packages
# ============================================================
# Run this cell in a fresh kernel before importing MLflow.
%pip install -q -U \
    "boto3" \
    "botocore" \
    "mlflow==3.15.1" \
    "mlflow-skinny==3.15.1" \
    "mlflow-tracing==3.15.1" \
    "sagemaker-mlflow==0.5.0" \
    "imbalanced-learn" \
    "xgboost" \
    "pyarrow"


Note: you may need to restart the kernel to use updated packages.


## 1. Configuration, S3/local fallback, MLflow

In [2]:
# ============================================================
# Configuration, S3/local fallback, MLflow
# Mirrors the shared pattern used across EDA / Feature Engineering / Model A
# ============================================================
from pathlib import Path
import os
import json
import warnings
import boto3
from urllib.parse import urlparse
from datetime import datetime, timezone

warnings.filterwarnings("ignore")

# ---- Project storage configuration ----
S3_BUCKET_URI = os.getenv(
    "TEAM04_S3_BUCKET_URI",
    "s3://nyp-26s1-iti113/iti113/team04/data/credit-card-fraud-detection/"
).rstrip("/") + "/"

LOCAL_DATA_DIR = Path(os.getenv("TEAM04_LOCAL_DATA_DIR", "./data"))

# ---- The one parameter that changes between retraining runs ----
# Point this at the new data file when new transactions are available.
# Everything else in this notebook is driven from this single value.
NEW_DATA_FILENAME = os.getenv("TEAM04_RETRAIN_DATA_FILENAME", "fraudTrain.csv")

FE_OUTPUT_DIRNAME = "feature_engineering"
FE_CONFIG_FILENAME = "feature_engineering_manifest.json"
EDA_CONFIG_FILENAME = "eda_feature_candidates.json"
MODELA_OUTPUT_DIRNAME = "modela_baseline"

S3_RAW_URI = S3_BUCKET_URI + "raw/"
S3_FE_OUTPUT_URI = S3_BUCKET_URI + f"processed/{FE_OUTPUT_DIRNAME}/"
S3_MODELA_OUTPUT_URI = S3_BUCKET_URI + f"processed/{MODELA_OUTPUT_DIRNAME}/"

LOCAL_RAW_DIR = LOCAL_DATA_DIR / "raw"
LOCAL_FE_OUTPUT_DIR = LOCAL_DATA_DIR / FE_OUTPUT_DIRNAME
LOCAL_MODELA_OUTPUT_DIR = LOCAL_DATA_DIR / MODELA_OUTPUT_DIRNAME
LOCAL_RETRAIN_DIR = LOCAL_DATA_DIR / "modela_retrain"

for _p in [LOCAL_RAW_DIR, LOCAL_FE_OUTPUT_DIR, LOCAL_MODELA_OUTPUT_DIR, LOCAL_RETRAIN_DIR]:
    _p.mkdir(parents=True, exist_ok=True)

LOCAL_NEW_DATA_PATH = LOCAL_RAW_DIR / NEW_DATA_FILENAME
LOCAL_FE_CONFIG_PATH = LOCAL_FE_OUTPUT_DIR / FE_CONFIG_FILENAME
LOCAL_EDA_CONFIG_PATH = LOCAL_FE_OUTPUT_DIR / EDA_CONFIG_FILENAME


def s3_join(prefix, *parts):
    return prefix.rstrip("/") + "/" + "/".join(str(part).strip("/") for part in parts)


def parse_s3_uri(uri):
    p = urlparse(uri)
    if p.scheme != "s3" or not p.netloc:
        raise ValueError(f"Invalid S3 URI: {uri}")
    return p.netloc, p.path.lstrip("/")


def download_s3_to_local(s3_uri, local_path):
    """Try S3 first. If it fails, use an existing local file as failover."""
    local_path = Path(local_path)
    local_path.parent.mkdir(parents=True, exist_ok=True)
    bucket, key = parse_s3_uri(s3_uri)
    try:
        boto3.client("s3").download_file(bucket, key, str(local_path))
        print(f"✓ Loaded from S3: {s3_uri}")
        return local_path
    except Exception as exc:
        if local_path.exists():
            print(f"⚠ S3 unavailable; using local failover: {local_path}")
            print(f"  {type(exc).__name__}: {exc}")
            return local_path
        raise RuntimeError(
            f"S3 load failed and local failover does not exist: {local_path}\n"
            f"S3 URI: {s3_uri}\nOriginal error: {type(exc).__name__}: {exc}"
        ) from exc


def upload_local_to_s3(local_path, s3_uri):
    local_path = Path(local_path)
    bucket, key = parse_s3_uri(s3_uri)
    try:
        boto3.client("s3").upload_file(str(local_path), bucket, key)
        print(f"✓ Saved to S3: {s3_uri}")
        return True
    except Exception as exc:
        print(f"⚠ S3 upload failed; local output retained: {local_path}")
        print(f"  {type(exc).__name__}: {exc}")
        return False


def load_json_s3_or_local(s3_uri, local_path):
    local_path = download_s3_to_local(s3_uri, local_path)
    with open(local_path, "r", encoding="utf-8") as f:
        return json.load(f)


import mlflow
from mlflow import MlflowClient
from mlflow_utils import initialize_mlflow

STUDENT_ID = "S402"
EXPERIMENT_NAME = "ITI113/team04/ModelA"

MLFLOW_APP_ARN = initialize_mlflow(student_id=STUDENT_ID, experiment_name=EXPERIMENT_NAME)

REGISTERED_MODEL_NAME = "ITI113-team04-ModelA-XGBoost-Final"
CHAMPION_ALIAS = "champion-candidate"

print(f"MLflow experiment: {EXPERIMENT_NAME}")
print(f"Retraining data file: {NEW_DATA_FILENAME}")
print(f"Registered model: {REGISTERED_MODEL_NAME} (alias: {CHAMPION_ALIAS})")


Initializing SageMaker MLflow connection for S402...
Target Experiment: ITI113/team04/ModelA
MLflow App ARN: arn:aws:sagemaker:ap-southeast-1:044528205969:mlflow-app/app-ANFQ3RACFV2G
MLflow Tracking URI successfully set.
Fresh MLflow UI URL:
https://app-ANFQ3RACFV2G.mlflow.sagemaker.ap-southeast-1.app.aws/auth?authToken=eyJhbGciOiJIUzI1NiJ9.eyJhdXRoVG9rZW5JZCI6IlNZU1pGSCIsImZhc0NyZWRlbnRpYWxzIjoiQWdWNFR3QTBXZTJJeHlFRllLd0dQbS9tRHE2d2lHTVgxYVJ6U2ltU3hYRHFkM2dBWHdBQkFCVmhkM010WTNKNWNIUnZMWEIxWW14cFl5MXJaWGtBUkVFNVREUm1WMHRQVDIwMFZuTlZRMFJNYm1KUVpYRjVWa2N4TTFaUlZHODNlak5aZUdKbWRsaEpTRmhXU1ZsV1JsbFVWbUZ3ZFRkUloxTmtXbWQ0VEV4b1FUMDlBQUVBQjJGM2N5MXJiWE1BVUdGeWJqcGhkM002YTIxek9tRndMWE52ZFhSb1pXRnpkQzB4T2pNNU5qa3hNemN6TnpJMU5EcHJaWGt2WVRBNU1XRmhNRE10TnprMU5TMDBaakF5TFdJMVpHWXRaVE5oTlRNd1pXSmlaVGcxQUxnQkFnRUFlT0thVkkrUUdqak5TNEo0TUhCNk91SlA3UGFLdlRHSG9tY2kveDlrZTJiekFXMm5qaWdFbUp6L3hnWmxhTnp0aHZVQUFBQitNSHdHQ1NxR1NJYjNEUUVIQnFCdk1HMENBUUF3YUFZSktvWklodmNOQVFjQk1CNEdDV0NHU0FGbEF3UUJMakFSQkF6aEdiL

## 2. Load new raw data and Feature Engineering's config

`eda_feature_candidates.json` and `feature_engineering_manifest.json` are
read fresh every run — they are the source of truth for which features and
transforms apply. This notebook does not hard-code a feature list.

In [3]:
import pandas as pd
import numpy as np

RAW_S3_URI = s3_join(S3_RAW_URI, NEW_DATA_FILENAME)
FE_CONFIG_S3_URI = s3_join(S3_FE_OUTPUT_URI, FE_CONFIG_FILENAME)

raw_df = pd.read_csv(download_s3_to_local(RAW_S3_URI, LOCAL_NEW_DATA_PATH))
FE_CONFIG = load_json_s3_or_local(FE_CONFIG_S3_URI, LOCAL_FE_CONFIG_PATH)
SELECTED_FEATURES = FE_CONFIG["selected_features"]

print(f"Raw data: {raw_df.shape[0]:,} rows x {raw_df.shape[1]} columns")
print(f"Selected features (from Feature Engineering manifest): {SELECTED_FEATURES}")
print(f"Fraud rate: {raw_df['is_fraud'].mean():.4%}")


✓ Loaded from S3: s3://nyp-26s1-iti113/iti113/team04/data/credit-card-fraud-detection/raw/fraudTrain.csv
✓ Loaded from S3: s3://nyp-26s1-iti113/iti113/team04/data/credit-card-fraud-detection/processed/feature_engineering/feature_engineering_manifest.json
Raw data: 1,296,675 rows x 23 columns
Selected features (from Feature Engineering manifest): ['amt', 'amt_log', 'category_te', 'distance_km', 'distance_log', 'city_pop', 'trans_hour', 'day_of_week', 'is_weekend', 'age', 'gender_binary']
Fraud rate: 0.5789%


## 3. Re-apply Feature Engineering's transforms

These formulas mirror Feature Engineering Section 2 (non-target-dependent
transforms). **Diff this cell against that notebook before trusting it** —
see the warning in Section 0.

In [4]:
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    a = np.clip(a, 0.0, 1.0)  # guards against floating-point domain errors in arcsin
    return 2 * R * np.arcsin(np.sqrt(a))


model_df = raw_df.copy()

model_df["trans_date_trans_time"] = pd.to_datetime(model_df["trans_date_trans_time"], errors="coerce")
model_df["dob"] = pd.to_datetime(model_df["dob"], errors="coerce")

model_df["trans_hour"] = model_df["trans_date_trans_time"].dt.hour
model_df["day_of_week"] = model_df["trans_date_trans_time"].dt.dayofweek  # 0=Monday ... 6=Sunday
model_df["is_weekend"] = model_df["trans_date_trans_time"].dt.dayofweek.isin([5, 6]).astype(int)

model_df["age"] = (
    model_df["trans_date_trans_time"].dt.year - model_df["dob"].dt.year
    - (
        (model_df["trans_date_trans_time"].dt.month < model_df["dob"].dt.month)
        | (
            (model_df["trans_date_trans_time"].dt.month == model_df["dob"].dt.month)
            & (model_df["trans_date_trans_time"].dt.day < model_df["dob"].dt.day)
        )
    ).astype(int)
)

model_df["distance_km"] = haversine_km(
    model_df["lat"], model_df["long"], model_df["merch_lat"], model_df["merch_long"]
)
model_df["distance_log"] = np.log1p(model_df["distance_km"])
model_df["amt_log"] = np.log1p(model_df["amt"])
model_df["gender_binary"] = model_df["gender"].map({"F": 0, "M": 1})

_missing = set(SELECTED_FEATURES) - set(model_df.columns) - {"category_te"}
if _missing:
    raise AssertionError(
        f"Retraining transform is missing expected columns: {sorted(_missing)}. "
        "This means Section 3 has drifted from Feature Engineering's actual "
        "logic — diff against that notebook before proceeding."
    )

print(f"✓ Transforms applied. Shape: {model_df.shape[0]:,} rows x {model_df.shape[1]} columns")


✓ Transforms applied. Shape: 1,296,675 rows x 31 columns


## 4. Stratified split — identical methodology to the locked pipeline

Same 60/20/20 ratios and `random_state`. Nothing about the split strategy is
re-decided during retraining.

In [5]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
RAW_MODEL_COLUMNS = [c for c in SELECTED_FEATURES if c != "category_te"] + ["category", "gender", "is_fraud"]
RAW_MODEL_COLUMNS = list(dict.fromkeys(RAW_MODEL_COLUMNS))  # de-duplicate, preserve order

X = model_df[[c for c in RAW_MODEL_COLUMNS if c != "is_fraud"]]
y = model_df["is_fraud"]

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_temp, y_temp, test_size=0.25, stratify=y_temp, random_state=RANDOM_STATE
)

print(f"Train: {len(X_train):,} | Valid: {len(X_valid):,} | Test: {len(X_test):,}")
print(f"Fraud rates — train: {y_train.mean():.4%} | valid: {y_valid.mean():.4%} | test: {y_test.mean():.4%}")


Train: 778,005 | Valid: 259,335 | Test: 259,335
Fraud rates — train: 0.5789% | valid: 0.5788% | test: 0.5788%


## 5. Out-of-fold target encoding — identical methodology

Same K-fold OOF approach and smoothing as Feature Engineering and Model A.
Fit fresh on the new training data each retrain, since the category-level
fraud rates are exactly what's expected to shift as new data arrives.

In [6]:
from sklearn.model_selection import StratifiedKFold

N_SPLITS_TE = 5
SMOOTHING = 20


def fit_te_mapping(categories, target, smoothing=SMOOTHING):
    stats = target.groupby(categories).agg(["mean", "count"])
    global_mean = target.mean()
    smoothed = (stats["mean"] * stats["count"] + global_mean * smoothing) / (stats["count"] + smoothing)
    return smoothed.to_dict(), float(global_mean)


def apply_te(categories, mapping, global_mean):
    return categories.map(mapping).fillna(global_mean).astype(float)


def kfold_target_encode(categories, target, n_splits=N_SPLITS_TE, smoothing=SMOOTHING, random_state=RANDOM_STATE):
    encoded = pd.Series(index=categories.index, dtype=float)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    for fit_idx, hold_idx in skf.split(categories, target):
        fold_mapping, fold_global_mean = fit_te_mapping(
            categories.iloc[fit_idx], target.iloc[fit_idx], smoothing
        )
        encoded.iloc[hold_idx] = apply_te(categories.iloc[hold_idx], fold_mapping, fold_global_mean)
    return encoded


X_train = X_train.copy()
X_train["category_te"] = kfold_target_encode(X_train["category"], y_train)

te_mapping, te_global_mean = fit_te_mapping(X_train["category"], y_train)
X_valid = X_valid.copy()
X_test = X_test.copy()
X_valid["category_te"] = apply_te(X_valid["category"], te_mapping, te_global_mean)
X_test["category_te"] = apply_te(X_test["category"], te_mapping, te_global_mean)

X_train_model = X_train[SELECTED_FEATURES]
X_valid_model = X_valid[SELECTED_FEATURES]
X_test_model = X_test[SELECTED_FEATURES]

print(f"✓ Target encoding fit on {len(X_train):,} training rows, {len(te_mapping)} categories.")


✓ Target encoding fit on 778,005 training rows, 14 categories.


## 6. SMOTE (training data only) and retrain using the locked final configuration

In [7]:
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

smote = SMOTE(sampling_strategy=0.20, random_state=RANDOM_STATE)
X_train_smote, y_train_smote = smote.fit_resample(X_train_model, y_train)

# These are the FINAL, already-tuned hyperparameters locked in the Model A
# Final notebook (Section 14). Retraining re-fits on new data; it does not
# re-run hyperparameter search — copy the exact values from that notebook's
# XGB_PARAMS / FINAL_CANDIDATES winning entry before running this for real.
FINAL_XGB_PARAMS = {
    "n_estimators": 300,
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "objective": "binary:logistic",
    "eval_metric": "aucpr",
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "tree_method": "hist",
}

candidate_model = XGBClassifier(**FINAL_XGB_PARAMS)
candidate_model.fit(X_train_smote, y_train_smote, eval_set=[(X_valid_model, y_valid)], verbose=False)

print("✓ Candidate model retrained on new data using the locked final configuration.")


✓ Candidate model retrained on new data using the locked final configuration.


## 7. Select threshold on validation, evaluate candidate on held-out test

Same validation-only threshold selection discipline as every other notebook
in this project.

In [8]:
from sklearn.metrics import (
    average_precision_score, roc_auc_score, precision_score,
    recall_score, f1_score, confusion_matrix
)

valid_prob = candidate_model.predict_proba(X_valid_model)[:, 1]

threshold_rows = []
for threshold in np.arange(0.05, 1.001, 0.01):
    pred = (valid_prob >= threshold).astype(int)
    threshold_rows.append({
        "threshold": round(float(threshold), 2),
        "precision": precision_score(y_valid, pred, zero_division=0),
        "recall": recall_score(y_valid, pred, zero_division=0),
        "f1": f1_score(y_valid, pred, zero_division=0),
    })
threshold_df = pd.DataFrame(threshold_rows)
best_row = threshold_df.loc[threshold_df["f1"].idxmax()]
CANDIDATE_THRESHOLD = float(best_row["threshold"])

test_prob = candidate_model.predict_proba(X_test_model)[:, 1]
test_pred = (test_prob >= CANDIDATE_THRESHOLD).astype(int)

candidate_metrics = {
    "pr_auc": float(average_precision_score(y_test, test_prob)),
    "roc_auc": float(roc_auc_score(y_test, test_prob)),
    "precision": float(precision_score(y_test, test_pred, zero_division=0)),
    "recall": float(recall_score(y_test, test_pred, zero_division=0)),
    "f1": float(f1_score(y_test, test_pred, zero_division=0)),
}

print(f"Candidate threshold: {CANDIDATE_THRESHOLD}")
print("Candidate test metrics:", json.dumps(candidate_metrics, indent=2))


Candidate threshold: 0.94
Candidate test metrics: {
  "pr_auc": 0.8133461030179584,
  "roc_auc": 0.9904491740878453,
  "precision": 0.8320493066255779,
  "recall": 0.7195203197868087,
  "f1": 0.7717041800643086
}


## 8. Compare against the currently deployed model — promote only if better

This is the piece that's new relative to a one-off training run: fetch
whatever the `champion-candidate` alias currently points at, pull its
recorded test metrics, and only register + move the alias if the retrained
candidate genuinely beats it. If not, the comparison is still logged — the
retrain isn't wasted, it's evidence the current champion still holds — but
nothing currently deployed changes.

In [9]:
from mlflow.exceptions import MlflowException
from mlflow import MlflowClient

client = MlflowClient()

PROMOTION_METRIC = "test_pr_auc"  # the metric that decides promotion
current_champion_metrics = None
current_champion_version = None
current_champion = None

# 1. Fetch the current champion with a strict network/infrastructure safety gate
try:
    current_champion = client.get_model_version_by_alias(REGISTERED_MODEL_NAME, CHAMPION_ALIAS)
except MlflowException as exc:
    if "not found" in str(exc).lower() or "does not exist" in str(exc).lower():
        current_champion = None
        print(f"No champion registered yet under alias '{CHAMPION_ALIAS}' — treating as first registration.")
    else:
        # A genuine infrastructure error occurred (e.g., network timeout, IAM permission denied).
        # We MUST raise the error to prevent the pipeline from falsely assuming no champion exists.
        raise  

if current_champion is not None:
    current_champion_version = current_champion.version
    current_run = client.get_run(current_champion.run_id)
    current_champion_metrics = current_run.data.metrics
    print(f"Current champion: version {current_champion_version}")

candidate_pr_auc = candidate_metrics["pr_auc"]

# 2. Strict, Fail-Closed Promotion Metric Logic
if current_champion_metrics is None:
    # Scenario A: First ever run. No champion exists to compare against.
    current_pr_auc = None
    SHOULD_PROMOTE = True
else:
    # Scenario B: Champion exists. It MUST have the exact promotion metric logged.
    if PROMOTION_METRIC not in current_champion_metrics:
        raise ValueError(
            f"CRITICAL MLOPS FAILURE: The current champion (version {current_champion_version}) "
            f"is missing the required metric '{PROMOTION_METRIC}'. "
            f"Safety check disabled. Halting pipeline to prevent unsafe promotion."
        )
    
    current_pr_auc = current_champion_metrics[PROMOTION_METRIC]
    SHOULD_PROMOTE = candidate_pr_auc > current_pr_auc

print()
print(f"Candidate {PROMOTION_METRIC}: {candidate_pr_auc:.4f}")
print(f"Current champion {PROMOTION_METRIC}: {current_pr_auc if current_pr_auc is not None else 'n/a'}")
print(f"Decision: {'PROMOTE — candidate is better' if SHOULD_PROMOTE else 'DO NOT PROMOTE — keep current champion'}")

Current champion: version 5

Candidate test_pr_auc: 0.8133
Current champion test_pr_auc: 0.8347904369579984
Decision: DO NOT PROMOTE — keep current champion


## 9. Log the run; register and promote only if the decision says so

The retrain is always logged to MLflow — win or lose — so the comparison
history itself is an audit trail. Registration and the alias move only
happen inside the `if SHOULD_PROMOTE:` branch.

In [10]:
RUN_NAME = f"{STUDENT_ID}_MODELA_RETRAIN_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"

if mlflow.active_run():
    mlflow.end_run()

with mlflow.start_run(run_name=RUN_NAME) as run:
    mlflow.set_tags({
        "Course": "ITI113", "Semester": "26S1", "TeamId": "team04",
        "StudentId": STUDENT_ID, "ProjectName": "credit-card-fraud-detection",
        "CreatedByNotebook": "ModelA_Retrain",
        "PipelineStage": "ModelA_Retrain",
        "PromotionDecision": "promoted" if SHOULD_PROMOTE else "not_promoted",
    })
    mlflow.log_params(FINAL_XGB_PARAMS)
    mlflow.log_param("retrain_data_filename", NEW_DATA_FILENAME)
    mlflow.log_param("candidate_threshold", CANDIDATE_THRESHOLD)
    mlflow.log_param("compared_against_version", current_champion_version)

    for k, v in candidate_metrics.items():
        mlflow.log_metric(f"test_{k}", v)
    if current_pr_auc is not None:
        mlflow.log_metric("champion_test_pr_auc_at_comparison", current_pr_auc)

    input_example = X_train_model.head(5)
    from mlflow.models import infer_signature
    signature = infer_signature(input_example, candidate_model.predict_proba(input_example))

    mlflow.xgboost.log_model(
        candidate_model, artifact_path="model",
        input_example=input_example, signature=signature,
    )

    if SHOULD_PROMOTE:
        registered_model = mlflow.register_model(
            model_uri=f"runs:/{run.info.run_id}/model",
            name=REGISTERED_MODEL_NAME,
        )
        client.set_registered_model_alias(
            name=REGISTERED_MODEL_NAME, alias=CHAMPION_ALIAS, version=registered_model.version,
        )
        client.set_model_version_tag(REGISTERED_MODEL_NAME, registered_model.version, "promoted_from_run", run.info.run_id)

        # --------------------------------------------------------------
        # Export the same artifact set inference.py depends on, so a
        # promoted retrain updates them the same way Model A Final did.
        # Only done here, inside SHOULD_PROMOTE -- a losing candidate must
        # never overwrite the artifacts describing the actual champion.
        # --------------------------------------------------------------
        import joblib

        PREPROCESSING_BUNDLE_FILENAME = "model_a_preprocessing_bundle.joblib"
        DEPLOYMENT_CONTRACT_FILENAME = "model_a_deployment_contract.json"
        RETRAIN_RESULTS_FILENAME = "model_a_retrain_results.json"

        LOCAL_PREPROCESSING_BUNDLE_PATH = LOCAL_MODELA_OUTPUT_DIR / PREPROCESSING_BUNDLE_FILENAME
        LOCAL_DEPLOYMENT_CONTRACT_PATH = LOCAL_MODELA_OUTPUT_DIR / DEPLOYMENT_CONTRACT_FILENAME
        LOCAL_RETRAIN_RESULTS_PATH = LOCAL_MODELA_OUTPUT_DIR / RETRAIN_RESULTS_FILENAME

        preprocessing_bundle = {
            "category_te_mapping": te_mapping,
            "category_te_global_mean": te_global_mean,
            "gender_mapping": {"F": 0, "M": 1},
            "operating_threshold": CANDIDATE_THRESHOLD,
        }
        joblib.dump(preprocessing_bundle, LOCAL_PREPROCESSING_BUNDLE_PATH)
        upload_local_to_s3(LOCAL_PREPROCESSING_BUNDLE_PATH, s3_join(S3_MODELA_OUTPUT_URI, PREPROCESSING_BUNDLE_FILENAME))

        # category_te / gender_binary map back to their raw pre-encoded field
        # names, so this stays correct even if SELECTED_FEATURES changes later.
        raw_input_fields_required = [
            {"category_te": "category", "gender_binary": "gender"}.get(c, c)
            for c in SELECTED_FEATURES
        ]
        deployment_contract = {
            "feature_columns": SELECTED_FEATURES,
            "raw_input_fields_required": raw_input_fields_required,
            "operating_threshold": CANDIDATE_THRESHOLD,
            "decision_rule": "fraud if predicted_probability >= operating_threshold",
            "source_run_id": run.info.run_id,
            "registered_model_version": registered_model.version,
        }
        with open(LOCAL_DEPLOYMENT_CONTRACT_PATH, "w", encoding="utf-8") as f:
            json.dump(deployment_contract, f, indent=2)
        upload_local_to_s3(LOCAL_DEPLOYMENT_CONTRACT_PATH, s3_join(S3_MODELA_OUTPUT_URI, DEPLOYMENT_CONTRACT_FILENAME))

        retrain_results = {
            "schema_version": "1.0",
            "generated_at_utc": datetime.now(timezone.utc).isoformat(),
            "run_id": run.info.run_id,
            "registered_model_version": registered_model.version,
            "retrain_data_filename": NEW_DATA_FILENAME,
            "candidate_threshold": CANDIDATE_THRESHOLD,
            "candidate_metrics": candidate_metrics,
            "compared_against_version": current_champion_version,
            "champion_test_pr_auc_at_comparison": current_pr_auc,
            "promoted": True,
        }
        with open(LOCAL_RETRAIN_RESULTS_PATH, "w", encoding="utf-8") as f:
            json.dump(retrain_results, f, indent=2)
        upload_local_to_s3(LOCAL_RETRAIN_RESULTS_PATH, s3_join(S3_MODELA_OUTPUT_URI, RETRAIN_RESULTS_FILENAME))

        # --------------------------------------------------------------
        # Export test-set predictions for the Fairness Audit notebook,
        # using the SAME fixed S3 path every promoted model writes to --
        # this is what lets the fairness notebook always find "whichever
        # model is currently the champion" without needing reconfiguring.
        # --------------------------------------------------------------
        FAIRNESS_PREDICTIONS_FILENAME = "model_a_fairness_test_predictions.parquet"
        LOCAL_FAIRNESS_PREDICTIONS_PATH = LOCAL_MODELA_OUTPUT_DIR / FAIRNESS_PREDICTIONS_FILENAME

        fairness_audit_df = X_test.copy()
        fairness_audit_df["y_true"] = np.asarray(y_test)
        fairness_audit_df["y_prob"] = test_prob
        fairness_audit_df["y_pred"] = test_pred
        fairness_audit_df.to_parquet(LOCAL_FAIRNESS_PREDICTIONS_PATH, index=False)
        upload_local_to_s3(LOCAL_FAIRNESS_PREDICTIONS_PATH, s3_join(S3_MODELA_OUTPUT_URI, FAIRNESS_PREDICTIONS_FILENAME))
        mlflow.log_artifact(str(LOCAL_FAIRNESS_PREDICTIONS_PATH), artifact_path="deployment")

        mlflow.log_artifact(str(LOCAL_PREPROCESSING_BUNDLE_PATH), artifact_path="deployment")
        mlflow.log_artifact(str(LOCAL_DEPLOYMENT_CONTRACT_PATH), artifact_path="deployment")
        mlflow.log_artifact(str(LOCAL_RETRAIN_RESULTS_PATH), artifact_path="model_selection")

        print(f"✓ Promoted: version {registered_model.version} is now '{CHAMPION_ALIAS}'.")
        print(f"✓ Preprocessing bundle, deployment contract, and results manifest updated for this version.")
    else:
        registered_model = None
        print(f"✓ Logged for audit trail. Current champion (version {current_champion_version}) unchanged.")
        print("  No artifacts updated -- current champion's contract and bundle remain authoritative.")

    print(f"Run ID: {run.info.run_id}")


2026/08/18 13:54:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


✓ Logged for audit trail. Current champion (version 5) unchanged.
  No artifacts updated -- current champion's contract and bundle remain authoritative.
Run ID: 9d66a34eb4154b86969c8d513479b264
🏃 View run S402_MODELA_RETRAIN_20260818_135423 at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/4/runs/9d66a34eb4154b86969c8d513479b264
🧪 View experiment at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/4


### Closing note — the real long-term fix

Section 3 and Section 5 above reconstruct logic that already exists in your
Feature Engineering and Model A Final notebooks, specifically flagged for
manual diffing because copy-pasted logic across notebooks is exactly what
causes silent drift. The durable fix, if this project continues past this
milestone, is to move `haversine_km`, `fit_te_mapping`, `apply_te`, and
`kfold_target_encode` into one shared `feature_transforms.py` module that
every notebook — Feature Engineering, Model A Final, and this retraining
notebook — imports from, so there is exactly one place these formulas can
be edited, and every notebook using them updates together.
